# debt present value

## Setup

In [120]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parameter

In [121]:
par = 500000
coupon_rate = 3 / 100
months = 36 # maturity in months
freq = 1
ytm = 5.0 / 100
tax_rate = 15.0 / 100

## Cash flow

In [122]:
def periods(months, freq):
    return np.arange(1, int(round(months * freq / 12)) + 1)


def cash_flows(par, coupon_rate, months, freq):
    coupon = par * coupon_rate / freq
    flows = np.full(len(periods(months, freq)), coupon)
    flows[-1] += par
    return flows


def after_tax_cash_flows(par, coupon_rate, months, freq, tax_rate):
    coupon = par * coupon_rate / freq
    return cash_flows(par, coupon_rate, months, freq) - coupon * tax_rate

## Discount factor

In [123]:
def discount_factors(ytm, months, freq):
    return (1 + ytm / freq) ** -periods(months, freq)

## Present value

In [124]:
def present_value(par, coupon_rate, months, ytm, freq=1, tax_rate=0.0):
    flows = after_tax_cash_flows(par, coupon_rate, months, freq, tax_rate)
    factors = discount_factors(ytm, months, freq)
    return float(flows @ factors)

In [125]:
price = present_value(par, coupon_rate, months, ytm, freq)
price_after_tax = present_value(par, coupon_rate, months, ytm, freq, tax_rate)
pv_tax = price - price_after_tax

price, price_after_tax, pv_tax

(472767.51970629516, 466640.2116402116, 6127.308066083584)

## Schedule

In [126]:
schedule = pd.DataFrame({
    "period": periods(months, freq),
    "cash_flow": cash_flows(par, coupon_rate, months, freq),
    "after_tax_cash_flow": after_tax_cash_flows(par, coupon_rate, months, freq, tax_rate),
    "discount_factor": discount_factors(ytm, months, freq),
})
schedule["present_value"] = schedule["cash_flow"] * schedule["discount_factor"]
schedule["after_tax_present_value"] = schedule["after_tax_cash_flow"] * schedule["discount_factor"]
schedule.round(2)

,period,cash_flow,after_tax_cash_flow,discount_factor,present_value,after_tax_present_value
0,1,15000.0,12750.0,0.95,14285.71,12142.86
1,2,15000.0,12750.0,0.91,13605.44,11564.63
2,3,515000.0,512750.0,0.86,444876.36,442932.73


## Price and yield

In [127]:
curve = pd.DataFrame({"ytm": np.arange(0.02, 0.0801, 0.005)})
curve["price"] = [present_value(par, coupon_rate, months, y, freq) for y in curve["ytm"]]
curve["price_after_tax"] = [present_value(par, coupon_rate, months, y, freq, tax_rate) for y in curve["ytm"]]
curve.round({"ytm": 4, "price": 2, "price_after_tax": 2})

,ytm,price,price_after_tax
0,0.020,514419.42,507930.68
1,0.025,507140.06,500714.01
2,0.030,500000.00,493635.62
3,0.035,492995.91,486692.22
4,0.040,486124.54,479880.59
5,0.045,479382.77,473197.60
6,0.050,472767.52,466640.21
7,0.055,466275.83,460205.48
8,0.060,459904.82,453890.54
9,0.065,453651.68,447692.61


## IRR

In [128]:
def net_present_value(price, par, coupon_rate, months, rate, freq=1, tax_rate=0.0):
    return present_value(par, coupon_rate, months, rate, freq, tax_rate) - price


def irr(price, par, coupon_rate, months, freq=1, tax_rate=0.0, tol=1e-12):
    low, high = -freq + 1e-9, 1.0
    while high - low > tol:
        mid = (low + high) / 2
        if net_present_value(price, par, coupon_rate, months, mid, freq, tax_rate) > 0:
            low = mid
        else:
            high = mid
    return (low + high) / 2

In [129]:
market_price = 991000

irr_pre_tax = irr(market_price, par, coupon_rate, months, freq)
irr_after_tax = irr(market_price, par, coupon_rate, months, freq, tax_rate)

irr_pre_tax, irr_after_tax

(-0.18462963706701246, -0.1875120225992381)

In [130]:
quotes = pd.DataFrame({"price": np.arange(850_000, 1_150_001, 50_000)})
quotes["irr"] = [irr(p, par, coupon_rate, months, freq) for p in quotes["price"]]
quotes["irr_after_tax"] = [irr(p, par, coupon_rate, months, freq, tax_rate) for p in quotes["price"]]
quotes.round({"irr": 4, "irr_after_tax": 4})

,price,irr,irr_after_tax
0,850000,-0.1409,-0.1441
1,900000,-0.1575,-0.1605
2,950000,-0.1728,-0.1758
3,1000000,-0.1871,-0.1900
4,1050000,-0.2005,-0.2033
5,1100000,-0.2130,-0.2157
6,1150000,-0.2248,-0.2275
